# Installed Base Reliability and Service Cost Forecasting
## Part 3: Fleet Cost Forecast, Backtested

### Where we are
| Part | Output this notebook uses |
|---|---|
| 1 | `component_lifetimes.csv`: every component lifetime, failure or censored |
| 2 | Weibull failure models per component (2 or 3 parameter), Cox PH (Proportional Hazards) hazard ratios with a validation verdict |

Part 2 ended with a caveat: its next quarter number counted at most one failure per part and ignored preventive swaps. This notebook removes both simplifications, turns failures into dollars, and then asks the question that decides whether anyone should trust the model: **how well would it have forecast quarters we already know?**

### The business question
> What will servicing the installed base cost next quarter, how uncertain is that number, and how many spare parts should be on the shelf?

These map to the three decisions from Part 1: contract pricing (expected cost), financial reserves (the upper percentiles), and parts stocking (the demand quantile).

### Why simulate instead of multiplying a rate (numerical example)
Take one component position that just received a new part, with γ = 20 days, β = 1.5, η = 120 days, and no preventive swaps:

| Method | Expected failures in 90 days |
|---|---|
| First failure only (Part 2) | 1 − S(90) = **0.360** |
| With renewal: a failed part is replaced and the new one can fail too | **0.378** (simulated) |

Renewal adds 5% here. Preventive swaps work in the opposite direction: an old part swapped before it fails resets to age 0, where (with wear out and a failure free period) it is far less likely to fail. Which effect wins depends on the fleet's age mix and swap habits, so we simulate both instead of guessing.

### Approach
1. **Two competing clocks per part.** Every installed part has a failure clock and a preventive swap clock. Whichever rings first ends the part's life; the replacement starts at age 0 with fresh clocks. The failure clock is tried two ways, as Part 2's Weibull and as the empirical failure distribution, and the backtest picks.
2. **Monte Carlo.** Simulate the quarter thousands of times for every installed part. Each run gives failure counts, swap counts, and cost.
3. **Backtest with rolling origins.** Pretend it is day 150, 180, 210, 240, and 275 of 2015. At each origin, refit everything using only data available then, forecast the next 90 days, and compare with what actually happened. Compare against simple baselines to measure FVA (Forecast Value Added).
4. **Forward forecast.** Refit on all data and forecast the quarter after the data ends, in failures, parts, and dollars.

## 1. Setup

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import minimize

DATA_RAW = Path("data/raw")
DATA_PROCESSED = Path("data/processed")

COMPONENTS = ["comp1", "comp2", "comp3", "comp4"]
COLORS = {"comp1": "#534AB7", "comp2": "#D85A30", "comp3": "#1D9E75", "comp4": "#BA7517"}
RNG = np.random.default_rng(7)

HORIZON = 90          # forecast horizon in days (one quarter)
N_SIM = 4_000         # Monte Carlo runs per forecast
AIC_THRESHOLD = 2     # 3 parameter Weibull must beat 2 parameter by this much

plt.rcParams.update({"figure.figsize": (9, 4), "axes.grid": True, "grid.alpha": 0.3})

## 2. Load inputs
The lifetime table gets the same `entry` column as Part 2 (the age at which each part entered observation, for left truncation) and a second event flag, `prev_event`, marking lifetimes that ended in a preventive swap.

In [ ]:
lifetimes = pd.read_csv(DATA_PROCESSED / "component_lifetimes.csv", parse_dates=["start", "end"])
telemetry_bounds = pd.read_csv(DATA_RAW / "PdM_telemetry.csv", usecols=["datetime"], parse_dates=["datetime"])
OBS_START, OBS_END = telemetry_bounds["datetime"].min(), telemetry_bounds["datetime"].max()
del telemetry_bounds

lifetimes["entry"] = ((OBS_START - lifetimes["start"]).dt.total_seconds() / 86_400).clip(lower=0)
lifetimes = lifetimes[lifetimes["duration_days"] > lifetimes["entry"]].reset_index(drop=True)
lifetimes["prev_event"] = (lifetimes["ended_by"] == "preventive").astype(int)

cox_validation = pd.read_csv(DATA_PROCESSED / "cox_validation.csv", index_col="comp")
cox_hr = pd.read_csv(DATA_PROCESSED / "cox_hazard_ratios.csv")
machine_usage = pd.read_csv(DATA_PROCESSED / "machine_usage_h1.csv", index_col="machineID")

print(f"Window: {OBS_START.date()} to {OBS_END.date()}  |  {len(lifetimes):,} lifetimes  |  "
      f"{lifetimes['event'].sum():,} failures  |  {lifetimes['prev_event'].sum():,} preventive swaps")
cox_validation[["cv_mean_C", "verdict"]]

## 3. Reliability functions
These are the Part 2 functions, repeated so this notebook runs on its own (in the final repository they move to a shared `src/reliability.py` module). `fit_best` fits both the 2 and 3 parameter Weibull and keeps the 3 parameter version only under Part 2's rule: AIC (Akaike Information Criterion) improves by more than 2, β ≥ 1, and γ is clearly inside its range rather than pinned at the earliest event. Every refit in the backtest makes the same choice Part 2 made, automatically.

`kaplan_meier` is also Part 2's function; section 4 uses it for the swap clock.

In [ ]:
def weibull_S(t, beta, eta, gamma=0.0):
    z = np.clip(np.asarray(t, float) - gamma, 0, None) / eta
    return np.exp(-z ** beta)


def _unpack(x, gamma_max):
    beta, eta = np.exp(x[0]), np.exp(x[1])
    gamma = gamma_max / (1 + np.exp(-x[2])) if len(x) == 3 else 0.0
    return beta, eta, gamma


def weibull_negloglik(x, t, event, entry, gamma_max):
    beta, eta, gamma = _unpack(x, gamma_max)
    z = np.clip(t - gamma, 1e-12, None) / eta
    z_entry = np.clip(entry - gamma, 0, None) / eta
    log_h = np.log(beta / eta) + (beta - 1) * np.log(z)
    return -(np.sum(event * log_h) - np.sum(z ** beta) + np.sum(z_entry ** beta))


def fit_weibull(t, event, entry, location=False) -> dict:
    t, event, entry = (np.asarray(a, float) for a in (t, event, entry))
    gamma_max = 0.99 * t[event == 1].min()
    x0 = [0.0, np.log(t.mean())] + ([0.0] if location else [])
    res = minimize(weibull_negloglik, x0, args=(t, event, entry, gamma_max), method="Nelder-Mead",
                   options={"maxiter": 5_000, "xatol": 1e-6, "fatol": 1e-8})
    if not res.success:
        raise RuntimeError(res.message)
    beta, eta, gamma = _unpack(res.x, gamma_max)
    return {"beta": beta, "eta": eta, "gamma": gamma, "gamma_max": gamma_max, "AIC": 2 * len(x0) + 2 * res.fun}


def fit_best(t, event, entry) -> dict:
    f2 = fit_weibull(t, event, entry, location=False)
    try:
        f3 = fit_weibull(t, event, entry, location=True)
    except RuntimeError:
        return {**f2, "model": "2p"}
    valid_3p = f3["beta"] >= 1 and f3["gamma"] < 0.95 * f3["gamma_max"]
    if f2["AIC"] - f3["AIC"] > AIC_THRESHOLD and valid_3p:
        return {**f3, "model": "3p"}
    return {**f2, "model": "2p"}

def kaplan_meier(durations, events, entry=None) -> pd.DataFrame:
    d = np.asarray(durations, float)
    e = np.asarray(events, int)
    en = np.zeros_like(d) if entry is None else np.asarray(entry, float)
    times = np.unique(d[e == 1])
    at_risk = np.array([((en < t) & (d >= t)).sum() for t in times])
    deaths = np.array([((d == t) & (e == 1)).sum() for t in times])
    return pd.DataFrame({"time": times, "survival": np.cumprod(1 - deaths / at_risk)})

## 4. The clocks, and a property of this data that shapes them
A preventive swap ends a part's life just as surely as a failure does, so the simulation needs a clock for each. Each clock is estimated with the other event treated as censoring: for the swap clock, preventive swaps are the events and failures are censoring (the part left service before a swap could happen), and the reverse for the failure clock.

### Everything happens on a 15 day schedule
Look at how long parts lasted before each kind of event. The check below counts how many lifetimes are exact multiples of 15 days.

In [ ]:
ended = lifetimes[lifetimes["ended_by"].isin(["failure", "preventive"])]
schedule = (ended.groupby(["comp", "ended_by"])["duration_days"]
            .agg(events="size", share_multiple_of_15=lambda d: (d % 15 == 0).mean(),
                 most_common_days=lambda d: d.mode().iloc[0])
            .unstack("ended_by"))
schedule.round(3)

Most lifetimes end exactly on 15, 30, 45 days. Field service visits run on a fixed 15 day cycle, and both swaps and failures are **recorded at visits**. A part that breaks on day 38 is logged at the day 45 visit.

This matters for the simulation. A Weibull is a smooth curve: between visits it keeps producing failures that the real process only registers at the next visit, and around the first visits it cannot reproduce the near total absence of early failures followed by a steep climb. Part 2's Weibull remains the right tool for describing **how** components age (shape, B10, wear out); the question here is which clock forecasts counts best.

### Two versions of the failure clock, one of the swap clock
1. **Swap clock: empirical.** Swaps are purely a scheduling decision, so the Kaplan Meier estimate of time to swap is used directly as the distribution to sample from. It reproduces the schedule exactly. A Weibull forced onto it gives a degenerate fit (β far below 1, γ pinned at 0.99 × 15 = 14.85), the pattern Part 2's safeguard rejects.
2. **Failure clock, version A: Weibull** (Part 2's model, with its safeguard).
3. **Failure clock, version B: empirical** (Kaplan Meier of time to failure, swaps as censoring).

The backtest runs the simulation with **both** failure clocks and the forward forecast uses whichever scored better. That decision is made by data the model never saw, not by preference.

### Sampling a remaining time from an empirical clock
For a part of age a, with F the empirical probability of the event by a given age and m a hazard multiplier (1 for an average machine):

$$\text{event age} = F^{-1}\!\Big(1 - \big(1 - F(a)\big)\, U^{1/m}\Big)$$

With m = 1 this draws only from ages beyond a, in proportion to how often the event happened there. With m > 1 the conditional survival (1 − F(a + r)) / (1 − F(a)) is raised to the power m, exactly as a Cox hazard multiplier acts. Kaplan Meier may not reach 1 (some parts never had the event in the data); a draw that falls in that leftover probability means no event within the observed range.

### Numerical example
Suppose F(15) = 0.40, F(30) = 0.65, F(45) = 0.80 and m = 1. A part aged 20 days has F(20) = 0.40 (nothing happens between visits). With U = 0.5 the target is 1 − 0.60 × 0.5 = 0.70, which F first reaches at 45 days, so the event comes at age 45, 25 days from now.

The two clocks are treated as **independent competing risks**: knowing when a part would have been swapped tells us nothing extra about when it would have failed. That is an assumption, not a fact. If technicians swap parts that look worn, the clocks are linked, and the backtest is where a violation would show up as biased failure counts.

In [ ]:
def fit_empirical(t, event, entry) -> dict:
    km = kaplan_meier(t, event, entry)
    return {"model": "empirical", "times": km["time"].to_numpy(), "cdf": 1 - km["survival"].to_numpy()}


def fit_clocks(train: pd.DataFrame, fail_clock: str) -> dict:
    clocks = {}
    for comp in COMPONENTS:
        sub = train[train["comp"] == comp]
        t, entry = sub["duration_days"], sub["entry"]
        fail = fit_best(t, sub["event"], entry) if fail_clock == "weibull" else fit_empirical(t, sub["event"], entry)
        clocks[comp] = {"fail": fail, "prev": fit_empirical(t, sub["prev_event"], entry)}
    return clocks


def empirical_cdf_at(p: dict, age) -> np.ndarray:
    return np.concatenate([[0.0], p["cdf"]])[np.searchsorted(p["times"], age, side="right")]


def clock_cdf_at(p: dict, age: float) -> float:
    if p["model"] == "empirical":
        return float(empirical_cdf_at(p, age))
    return float(1 - weibull_S(age, p["beta"], p["eta"], p["gamma"]))


def clocks_table(clocks: dict) -> pd.DataFrame:
    rows = []
    for comp, c in clocks.items():
        row = {"comp": comp, "fail_model": c["fail"]["model"]}
        for d in (30, 45, 60, 90):
            row[f"P(failed by {d}d)"] = clock_cdf_at(c["fail"], d)
        for d in (15, 30, 90):
            row[f"P(swapped by {d}d)"] = clock_cdf_at(c["prev"], d)
        rows.append(row)
    return pd.DataFrame(rows).set_index("comp")


full_clocks = {kind: fit_clocks(lifetimes, kind) for kind in ("weibull", "empirical")}
pd.concat({kind: clocks_table(c) for kind, c in full_clocks.items()}).round(3)

Compare the two failure clocks row by row. Where the Weibull probability of having failed by 45 or 60 days sits well below the empirical one, the smooth curve is missing the steep climb after the first visits, and a simulation built on it will under forecast failures.

## 5. The simulation engine

### Sampling a remaining life from a Weibull clock
For a part of current age a, the remaining life r implied by the Weibull, given survival to a, can be drawn exactly with one uniform random number U:

$$a + r = \gamma + \eta \left[ x^{\beta} - \frac{\ln U}{m} \right]^{1/\beta}, \qquad x = \frac{\max(a - \gamma,\ 0)}{\eta}$$

m is the same hazard multiplier as above. Empirical clocks use the section 4 rule instead.

### One quarter, one run
For every installed part:
1. Draw a remaining time from the failure clock and one from the swap clock.
2. The earlier one happens first. If it falls inside the quarter, record a failure or a swap, reset the part to age 0, and draw again for the new part.
3. Stop when the next event falls outside the quarter.

With two empirical clocks, both events can land on the same visit. The data records only one, so a tie is resolved at random in proportion to the two clocks' hazards at that age, which is how the tie would split in the data.

The loop runs all parts and all simulations at once as NumPy arrays, so thousands of quarters take seconds.

In [ ]:
def empirical_hazard_at(p: dict, age) -> np.ndarray:
    F_before = np.concatenate([[0.0], p["cdf"]])[np.searchsorted(p["times"], age, side="left")]
    F_at = empirical_cdf_at(p, age)
    with np.errstate(divide="ignore", invalid="ignore"):
        return np.where(F_before < 1, (F_at - F_before) / (1 - F_before), 0.0)


def residual_life(age, p: dict, u, m=1.0):
    if p["model"] == "empirical":
        F_age = empirical_cdf_at(p, age)
        target = 1 - (1 - F_age) * u ** (1 / m)
        idx = np.searchsorted(p["cdf"], target, side="left")
        event_age = np.where(idx < len(p["times"]), p["times"][np.minimum(idx, len(p["times"]) - 1)], np.inf)
        return event_age - age
    x = np.clip(age - p["gamma"], 0, None) / p["eta"]
    return p["gamma"] + p["eta"] * (x ** p["beta"] - np.log(u) / m) ** (1 / p["beta"]) - age


def simulate_component(age: np.ndarray, clocks: dict, horizon: int = HORIZON, n_sim: int = N_SIM,
                       mult: np.ndarray | None = None) -> tuple[np.ndarray, np.ndarray]:
    n = len(age)
    m = np.ones(n) if mult is None else np.asarray(mult, float)
    A = np.tile(np.asarray(age, float), (n_sim, 1))     # current age of each part, per run
    t = np.zeros((n_sim, n))                            # time elapsed in the quarter
    active = np.ones((n_sim, n), bool)
    fails = np.zeros(n_sim, int)
    swaps = np.zeros(n_sim, int)
    both_empirical = clocks["fail"]["model"] == "empirical" and clocks["prev"]["model"] == "empirical"
    while active.any():
        r_fail = residual_life(A, clocks["fail"], RNG.random(A.shape), m)
        r_swap = residual_life(A, clocks["prev"], RNG.random(A.shape))
        dt = np.minimum(r_fail, r_swap)
        happens = active & (t + dt <= horizon)
        is_fail = r_fail < r_swap
        if both_empirical:
            tie = happens & (r_fail == r_swap)
            if tie.any():
                event_age = (A + r_fail)[tie]
                hf = empirical_hazard_at(clocks["fail"], event_age)
                hs = empirical_hazard_at(clocks["prev"], event_age)
                is_fail[tie] = RNG.random(tie.sum()) < hf / np.maximum(hf + hs, 1e-12)
        else:
            is_fail = r_fail <= r_swap
        is_fail &= happens
        fails += is_fail.sum(axis=1)
        swaps += (happens & ~is_fail).sum(axis=1)
        t = np.where(happens, t + dt, t)
        A = np.where(happens, 0.0, A)
        active = happens
    return fails, swaps


# Verify 1: the introduction's worked example (Weibull failure clock, swaps effectively never)
example = {"fail": {"model": "2p", "beta": 1.5, "eta": 120, "gamma": 20},
           "prev": {"model": "2p", "beta": 1.0, "eta": 1e12, "gamma": 0}}
f, _ = simulate_component(np.zeros(1), example, n_sim=200_000)
print(f"Expected failures with renewal: {f.mean():.3f}  (worked example: 0.378; first failure only: 0.360)")
assert abs(f.mean() - 0.378) < 0.01

# Verify 2: the empirical sampler reproduces the section 4 example
toy_clock = {"model": "empirical", "times": np.array([15.0, 30.0, 45.0]), "cdf": np.array([0.40, 0.65, 0.80])}
assert residual_life(np.array([20.0]), toy_clock, np.array([0.5]))[0] == 25.0
draws = residual_life(np.full(200_000, 20.0), toy_clock, RNG.random(200_000))
p_next_visit = np.mean(draws == 10.0)       # event at age 30: (0.65 - 0.40) / (1 - 0.40)
print(f"Empirical clock: P(event at next visit | age 20) = {p_next_visit:.3f}  (exact 0.417)")
assert abs(p_next_visit - 0.25 / 0.60) < 0.01

## 6. Rolling origin backtest

### Rebuilding history as of an origin
To forecast honestly from day 180, the model may only see what was known on day 180. For each origin:
1. Keep lifetimes that **started** before the origin.
2. Any lifetime still running at the origin is cut there and marked censored, even if we know it later failed. Using the true ending would leak the answer.
3. The **installed base** at the origin is every part in service at that moment, with age = origin − start. That includes parts fitted at the origin itself (age 0): if a visit happens exactly at the origin, the replaced part is gone and its successor is installed. Leaving those out would silently drop a few positions from every forecast.
4. Refit all clocks on this history, simulate the next 90 days with each failure clock, and count what actually happened in those 90 days from the full table.

### Baselines the model must beat
| Baseline | Forecast | What it ignores |
|---|---|---|
| Naive last quarter | failures in the 90 days before the origin | everything; pure persistence |
| Constant AFR (Annualized Failure Rate) | failures / exposure days × 90 × parts installed | age, renewal, swaps |
| Weibull first failure (Part 2 method) | Σ P(fail in 90 days given age) | renewal, swaps |
| **Simulation, Weibull failure clock** | Monte Carlo mean, 80% interval | the visit schedule, for failures |
| **Simulation, empirical failure clock** | Monte Carlo mean, 80% interval | nothing above |

In [ ]:
def history_as_of(lt: pd.DataFrame, origin: pd.Timestamp) -> pd.DataFrame:
    d = lt[lt["start"] < origin].copy()
    running = d["end"] > origin
    d.loc[running, "end"] = origin
    d.loc[running, ["event", "prev_event"]] = 0
    d.loc[running, "ended_by"] = "end_of_data"
    d["duration_days"] = (d["end"] - d["start"]).dt.total_seconds() / 86_400
    return d[d["duration_days"] > d["entry"]].reset_index(drop=True)


def installed_at(lt: pd.DataFrame, origin: pd.Timestamp) -> pd.DataFrame:
    # Parts in service at the origin, including replacements fitted at the origin itself (age 0)
    inst = lt[(lt["start"] <= origin) & (lt["end"] > origin)].copy()
    inst["age"] = (origin - inst["start"]).dt.total_seconds() / 86_400
    return inst


def actuals(lt: pd.DataFrame, origin: pd.Timestamp, horizon: int = HORIZON) -> pd.DataFrame:
    window = lt[(lt["end"] > origin) & (lt["end"] <= origin + pd.Timedelta(days=horizon))]
    return window.groupby("comp").agg(failures=("event", "sum"), swaps=("prev_event", "sum")).reindex(COMPONENTS, fill_value=0)


ORIGIN_DAYS = [150, 180, 210, 240, 275]
CLOCK_KINDS = ["weibull", "empirical"]
origins = [OBS_START + pd.Timedelta(days=d) for d in ORIGIN_DAYS]
assert origins[-1] + pd.Timedelta(days=HORIZON) <= OBS_END, "Last forecast must end inside the data"

rows = []
for origin in origins:
    hist = history_as_of(lifetimes, origin)
    clocks = {kind: fit_clocks(hist, kind) for kind in CLOCK_KINDS}
    actual = actuals(lifetimes, origin)
    last_q = actuals(lifetimes, origin - pd.Timedelta(days=HORIZON))

    for comp in COMPONENTS:
        h = hist[hist["comp"] == comp]
        installed = installed_at(lifetimes[lifetimes["comp"] == comp], origin)
        assert installed["machineID"].is_unique, "One part per machine and component"
        age = installed["age"].to_numpy()
        fp = clocks["weibull"][comp]["fail"]
        afr = h["event"].sum() / (h["duration_days"] - h["entry"]).sum() * 365
        p_first = 1 - weibull_S(age + HORIZON, fp["beta"], fp["eta"], fp["gamma"]) / weibull_S(age, fp["beta"], fp["eta"], fp["gamma"])

        row = {"origin": origin.date(), "comp": comp, "installed": len(age),
               "actual": actual.loc[comp, "failures"], "actual_swaps": actual.loc[comp, "swaps"],
               "naive_last_quarter": last_q.loc[comp, "failures"],
               "constant_AFR": afr * HORIZON / 365 * len(age), "weibull_first_failure": p_first.sum()}
        for kind in CLOCK_KINDS:
            fails, swaps = simulate_component(age, clocks[kind][comp])
            row.update({f"sim_{kind}": fails.mean(), f"sim_{kind}_p10": np.percentile(fails, 10),
                        f"sim_{kind}_p90": np.percentile(fails, 90), f"sim_{kind}_swaps": swaps.mean()})
        rows.append(row)

backtest = pd.DataFrame(rows)
for kind in CLOCK_KINDS:
    backtest[f"covered_{kind}"] = backtest["actual"].between(backtest[f"sim_{kind}_p10"], backtest[f"sim_{kind}_p90"])
backtest.round(1)

### Scoring

For every origin and component we compare each method's forecast with the actual failure count:

1. **MAE (Mean Absolute Error):** the typical miss in failures per component per quarter.
2. **Bias:** the average of forecast − actual. Positive means the method over forecasts.
3. **WAPE (Weighted Absolute Percentage Error):** total absolute error ÷ total actual failures. Unlike MAPE (Mean Absolute Percentage Error) it does not explode when a component has very few failures.
4. **FVA (Forecast Value Added):** how much the method cuts MAE relative to the naive baseline. If a sophisticated model cannot beat "same as last quarter", it has not earned its complexity.
5. **Coverage** of the 80% interval, for the two simulations.

### Numerical example of FVA
If naive has MAE = 8 failures and a simulation has MAE = 5, FVA = (8 − 5) / 8 = **37.5%**. At $19,600 expected per corrective event (the comp2 cost below), shrinking the typical miss by 3 failures shrinks the typical cost miss by about $59,000 per component per quarter.

In [ ]:
METHODS = ["naive_last_quarter", "constant_AFR", "weibull_first_failure", "sim_weibull", "sim_empirical"]


def score(df: pd.DataFrame) -> pd.DataFrame:
    out = {}
    for m in METHODS:
        err = df[m] - df["actual"]
        out[m] = {"MAE": err.abs().mean(), "bias": err.mean(), "WAPE": err.abs().sum() / df["actual"].sum()}
    s = pd.DataFrame(out).T
    s["FVA_vs_naive"] = 1 - s["MAE"] / s.loc["naive_last_quarter", "MAE"]
    for kind in CLOCK_KINDS:
        s.loc[f"sim_{kind}", "coverage_80"] = df[f"covered_{kind}"].mean()
    return s


scores = score(backtest)
FORECAST_CLOCK = min(CLOCK_KINDS, key=lambda k: scores.loc[f"sim_{k}", "MAE"])
print(f"Failure clock used for the forward forecast: {FORECAST_CLOCK} (lower backtest MAE)")
print(f"With {len(backtest)} forecasts, 80% coverage anywhere from about 60% to 95% is consistent with calibration.")
scores.round(3)

**Reading the scores.** Report these honestly whichever method wins; a backtest exists to find out, not to confirm.
1. **Weibull clock versus empirical clock.** If the empirical clock wins clearly, the visit schedule matters for counting failures, even though the Weibull describes ageing well. That is a useful distinction: one model for understanding, another for forecasting.
2. **Simulation versus naive.** Monthly failures in Part 1 were flat, so this fleet is in a steady state where "same as last quarter" is hard to beat. A simulation that only matches naive here still earns its place: it gives calibrated intervals (naive gives none), it reacts to changes in the fleet's age mix, swap policy, or installed base that persistence cannot see, and it produces parts and cost distributions from the same run.
3. **Bias.** A consistent under or over forecast by the best simulation usually means the independent clocks assumption is violated: if technicians swap parts that are about to fail, swaps steal failures that the model still counts. That would be a finding about the service process, worth a line in the README.
4. **Coverage near 80%** means the interval is honest: neither overconfident (far below 80%) nor padded (near 100%). Only calibrated intervals can be used for reserves.

The chart puts every backtest forecast next to what happened.

In [ ]:
fig, axes = plt.subplots(1, len(COMPONENTS), figsize=(16, 4), sharey=True)
x = np.arange(len(origins))
for ax, comp in zip(axes, COMPONENTS):
    sub = backtest[backtest["comp"] == comp]
    ax.fill_between(x, sub[f"sim_{FORECAST_CLOCK}_p10"], sub[f"sim_{FORECAST_CLOCK}_p90"], color=COLORS[comp],
                    alpha=0.2, label=f"simulation ({FORECAST_CLOCK}) 80% interval")
    ax.plot(x, sub[f"sim_{FORECAST_CLOCK}"], "o-", color=COLORS[comp], label=f"simulation ({FORECAST_CLOCK})")
    other = [k for k in CLOCK_KINDS if k != FORECAST_CLOCK][0]
    ax.plot(x, sub[f"sim_{other}"], "^:", color=COLORS[comp], alpha=0.6, label=f"simulation ({other})")
    ax.plot(x, sub["naive_last_quarter"], "s--", color="grey", ms=4, label="naive last quarter")
    ax.plot(x, sub["actual"], "k*", ms=11, label="actual")
    ax.set_xticks(x, [f"day {d}" for d in ORIGIN_DAYS], rotation=45, fontsize=8)
    ax.set(title=comp, xlabel="Forecast origin")
axes[0].set_ylabel("Failures in the next 90 days")
axes[0].legend(fontsize=7)
plt.tight_layout()
plt.show()

## 7. From failures to dollars

### Cost model
Each event consumes one part. The two kinds of event cost different amounts:

| Event | Cost components | Uncertainty |
|---|---|---|
| Preventive swap | part + planned labor | fixed |
| Failure (corrective) | part × (1 + expedite premium) + unplanned labor | lognormal spread around that median: overtime, travel, secondary damage |

**The dollar figures below are illustrative placeholders.** The Azure dataset has no cost data. In a real deployment these come from the parts catalog and field service labor records, and they live in a configuration file rather than in code.

### Numerical example (comp2)
Corrective median = 12,000 × 1.3 + 2,500 = **$18,100**. With a lognormal spread σ = 0.4, the expected corrective cost is 18,100 × e^(0.4²/2) = **$19,607**. A preventive swap costs 12,000 + 1,200 = **$13,200**. Each failure that a preventive swap could have prevented costs about $6,400 more, which is the number a preventive maintenance policy is optimizing.

In [ ]:
COSTS = pd.DataFrame({
    "part_cost":        {"comp1": 8_000, "comp2": 12_000, "comp3": 5_000, "comp4": 9_000},
    "labor_preventive": {"comp1": 1_200, "comp2": 1_200, "comp3": 1_200, "comp4": 1_200},
    "labor_corrective": {"comp1": 2_500, "comp2": 2_500, "comp3": 2_500, "comp4": 2_500},
})
EXPEDITE_PREMIUM = 0.30   # unplanned parts ship express
CORRECTIVE_SIGMA = 0.40   # lognormal spread of corrective cost

COSTS["preventive_cost"] = COSTS["part_cost"] + COSTS["labor_preventive"]
COSTS["corrective_median"] = COSTS["part_cost"] * (1 + EXPEDITE_PREMIUM) + COSTS["labor_corrective"]
COSTS["corrective_mean"] = COSTS["corrective_median"] * np.exp(CORRECTIVE_SIGMA ** 2 / 2)


def simulate_cost(fails: np.ndarray, swaps: np.ndarray, comp: str) -> np.ndarray:
    c = COSTS.loc[comp]
    run_of_event = np.repeat(np.arange(len(fails)), fails)
    draws = c["corrective_median"] * np.exp(CORRECTIVE_SIGMA * RNG.standard_normal(len(run_of_event)))
    corrective = np.bincount(run_of_event, weights=draws, minlength=len(fails))
    return corrective + swaps * c["preventive_cost"]


COSTS.round(0)

## 8. Machine level adjustment (only where it validated)
Part 2 fit Cox PH models per component and scored them with grouped cross validation. Here the rule is strict: a component's hazard ratios adjust individual machines **only if its verdict was `usable`**. Otherwise every machine gets multiplier 1, because multipliers that did not predict held out machines would add noise, not signal.

For a usable component, each machine's multiplier is exp(Σ coefficient × covariate), built from the same first half usage profile Part 2 used. The multipliers are then rescaled to average 1 across the installed base. The Weibull already reproduces the fleet average, so Cox should **redistribute** risk between machines, not inflate the total.

Numerical example: two machines with raw multipliers 1.5 and 0.5 already average 1, so they stay as they are: the first machine's parts fail as if they carried 1.5 times the hazard, the second's half of it, and the fleet total is unchanged.

In [ ]:
usage_z = (machine_usage - machine_usage.mean()) / machine_usage.std()
usage_z.columns = [f"{c}_z" for c in usage_z.columns]


def machine_multipliers(units: pd.DataFrame, comp: str) -> np.ndarray:
    if cox_validation.loc[comp, "verdict"] != "usable":
        return np.ones(len(units))
    coefs = cox_hr[cox_hr["comp"] == comp].set_index("covariate")["HR"].map(np.log)
    X = units[["machineID", "age", "model"]].merge(usage_z, left_on="machineID", right_index=True, how="left")
    for cov in coefs.index:
        if cov.startswith("model_"):
            X[cov] = (X["model"] == cov.removeprefix("model_")).astype(float)
    lin = X.reindex(columns=coefs.index).fillna(0).to_numpy() @ coefs.to_numpy()
    m = np.exp(lin)
    return m / m.mean()


print("Components adjusted by machine:", [c for c in COMPONENTS if cox_validation.loc[c, "verdict"] == "usable"] or "none")

**If no component is adjusted**, that is Part 2's validation doing its job: machine level usage profiles did not predict which machines fail, so the forecast treats every part of a component the same way. The fleet total is still well founded, because it comes from the Weibull clocks that the backtest above scored.

Note that the backtest evaluated the forecast **without** these multipliers. The Cox models were fit on second half lifetimes, which overlap the backtest quarters, so including them there would leak. Their out of sample evidence is the grouped cross validation in Part 2.

## 9. Forward forecast: the quarter after the data ends
Now use the clocks fitted on all available history, with the failure clock the backtest selected, and forecast the next 90 days for the parts installed at the end of the data.

**Parts demand** counts both failures and swaps, since both consume a part. Swaps are planned and ordered ahead; failures are what makes demand uncertain, so the stocking level is the 95th percentile of total demand. **Cost** percentiles support the reserve decision.

In [ ]:
# Parts in service when the data ends. A part replaced exactly at the last timestamp has a zero length
# successor that Part 1 could not record, so that position gets a fresh part at age 0.
last = lifetimes.sort_values("end").groupby(["machineID", "comp"]).tail(1)
fresh = last[(last["end"] == OBS_END) & (last["ended_by"] != "end_of_data")].assign(duration_days=0.0)
installed_now = pd.concat([lifetimes[lifetimes["ended_by"] == "end_of_data"], fresh], ignore_index=True)
assert (installed_now.groupby("comp")["machineID"].nunique() == lifetimes["machineID"].nunique()).all(), \
    "Every machine should have exactly one installed part per component"

forecast_rows, total_cost = [], np.zeros(N_SIM)
for comp in COMPONENTS:
    units = installed_now[installed_now["comp"] == comp]
    mult = machine_multipliers(units, comp)
    fails, swaps = simulate_component(units["duration_days"].to_numpy(), full_clocks[FORECAST_CLOCK][comp], mult=mult)
    cost = simulate_cost(fails, swaps, comp)
    total_cost += cost
    demand = fails + swaps
    forecast_rows.append({"comp": comp, "installed": len(units),
                          "failures_mean": fails.mean(), "failures_p10": np.percentile(fails, 10),
                          "failures_p90": np.percentile(fails, 90), "swaps_mean": swaps.mean(),
                          "parts_mean": demand.mean(), "parts_stock_p95": np.percentile(demand, 95),
                          "cost_mean": cost.mean(), "cost_p50": np.percentile(cost, 50),
                          "cost_p90": np.percentile(cost, 90), "cost_p95": np.percentile(cost, 95)})

forecast = pd.DataFrame(forecast_rows).set_index("comp")
forecast.loc["fleet"] = forecast.sum(numeric_only=True)
# Percentiles do not add across components: take fleet percentiles from the simulated fleet total
forecast.loc["fleet", ["failures_p10", "failures_p90", "parts_stock_p95"]] = np.nan
forecast.loc["fleet", ["cost_p50", "cost_p90", "cost_p95"]] = np.percentile(total_cost, [50, 90, 95])
forecast.round(0)

**Why the fleet percentiles are recomputed.** The P90 of the fleet is not the sum of the component P90s. All four components hitting a bad quarter at once is rarer than any one of them doing so, so adding P90s overstates the reserve. The fleet row takes its percentiles from the simulated fleet total instead: the diversification benefit of a larger installed base, in one line of code.

### The cost distribution and what each number decides

In [ ]:
mean, p90, p95 = total_cost.mean(), np.percentile(total_cost, 90), np.percentile(total_cost, 95)

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(total_cost / 1e3, bins=60, color="#534AB7", alpha=0.7)
for value, label, style in [(mean, "mean (pricing)", "-"), (p90, "P90 (reserve)", "--"), (p95, "P95", ":")]:
    ax.axvline(value / 1e3, color="black", ls=style, label=f"{label}: ${value / 1e3:,.0f}k")
ax.set(title="Simulated fleet service cost, next quarter", xlabel="Cost ($ thousands)", ylabel="Simulated quarters")
ax.legend()
plt.show()

print(f"Expected cost ${mean:,.0f}  |  reserve to P90 ${p90:,.0f}  |  buffer above expected ${p90 - mean:,.0f} "
      f"({(p90 - mean) / mean:.1%})")

| Number | Decision it supports |
|---|---|
| Mean | **Contract pricing.** The expected cost the contract price must cover before margin and risk loading. |
| P90 | **Reserves.** Holding this amount covers 9 quarters out of 10. The buffer above the mean is the price of that confidence. |
| `parts_stock_p95` | **Parts stocking.** Stocking this many parts per component meets all demand in about 19 quarters out of 20. |

The buffer is also a direct measure of forecast value: a model that narrows the distribution shrinks the reserve that must be held for the same confidence.

## 10. Save outputs

In [ ]:
backtest.to_csv(DATA_PROCESSED / "backtest_results.csv", index=False)
scores.to_csv(DATA_PROCESSED / "backtest_scores.csv")
forecast.to_csv(DATA_PROCESSED / "next_quarter_forecast.csv")
pd.concat({k: clocks_table(c) for k, c in full_clocks.items()}).to_csv(DATA_PROCESSED / "clock_params.csv")
print("Saved: backtest_results.csv, backtest_scores.csv, next_quarter_forecast.csv, clock_params.csv")

### Summary
1. Found that swaps and failures are both recorded on a **15 day service visit schedule**, and modeled every installed part with two competing clocks: an empirical swap clock, and a failure clock tried both as Part 2's Weibull and as an empirical distribution, with the backtest choosing. Renewals are simulated so a replaced part can fail again inside the quarter.
2. Verified the simulation engine and the empirical sampler against worked examples with known answers.
3. **Backtested** from five rolling origins, rebuilding history as of each origin so no future information leaked, and scored both simulations against naive, constant AFR, and first failure baselines with MAE, bias, WAPE, FVA, and interval coverage.
4. Converted failures and swaps into **cost distributions** with an explicit, replaceable cost model, and applied machine level Cox adjustments only where Part 2's validation allowed.
5. Produced the forward forecast in the three forms the business uses: expected cost for pricing, P90 for reserves, and a P95 parts quantity for stocking, with fleet percentiles computed from the simulated total rather than summed.

### Limitations to state in the README
1. Costs are illustrative placeholders.
2. Failure and swap clocks are assumed independent.
3. Empirical clocks cannot extrapolate beyond the oldest ages seen in the data; a fleet with much older parts would need the Weibull clock or a parametric tail.
4. The backtest covers five overlapping quarters within a single year, so interval coverage is estimated from 20 forecasts.
5. The installed base is fixed at 100 machines; a real forecast would add new installations and retirements.

### Next: Part 4, new generation forecasting
Everything so far relied on hundreds of failures per component. Part 4 asks what to do when a new platform has only a handful: a hierarchical Bayesian model that borrows strength from existing platforms, tested by pretending one platform is new and checking how quickly the forecast converges as its failures arrive.